In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import cross_val_score, cross_val_predict, StratifiedKFold
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
import os

In [11]:
df = pd.read_csv('Biometrics_4HZ/labeled_combined_biometric_data_4HZ.csv', low_memory=False)

In [12]:
df[['SPO2', 'pulse_rate']] = df[['SPO2', 'pulse_rate']].apply(pd.to_numeric, errors='coerce')

In [13]:
# Interpolate NaN values for SPO2 and Pulse_rate subject-session wise
df[['SPO2', 'pulse_rate']] = df.groupby(['user', 'session'])[['SPO2', 'pulse_rate']].apply(
    lambda group: group.interpolate(method='linear', limit_direction='forward')
).reset_index(drop=True)[['SPO2', 'pulse_rate']]

In [14]:
# Optionally save the clean data to a new CSV file

# Ensure the directory exists
output_dir = '4HZ_Individual_Data'
os.makedirs(output_dir, exist_ok=True)

# Save the DataFrame
df.to_csv(os.path.join(output_dir, 'final_biometric_4HZ.csv'), index=False)

In [15]:
df['DrowsinessLevel'] = df['label'].apply(lambda x: 0 if x < 4 else 1 if x<7 else 2)
df = df.drop(columns=['label'])


### Applying SVM for classification ###

In [ ]:
%%time
# --- Setup Cross-validation ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Separate features and target
X = df.drop(columns=['DrowsinessLevel', 'session', 'user'])
y = df['DrowsinessLevel']

# Apply Min-Max normalization
scaler = MinMaxScaler()
X_normalized = scaler.fit_transform(X)

# --- PCA Feature Selection ---
# PCA to retain a specific amount of variance
pca = PCA(n_components=0.90) 
X_pca = pca.fit_transform(X_normalized)

# Print the number of components selected and the explained variance ratio
print(f"Number of principal components selected: {X_pca.shape[1]}")
print(f"Explained variance ratio per component: {pca.explained_variance_ratio_}")
print(f"Cumulative explained variance: {np.cumsum(pca.explained_variance_ratio_)}")

# --- SVM ---
svm_clf = SVC(kernel='rbf', C=0.1, gamma='auto', cache_size=200,class_weight='balanced', random_state=42) 
svm_cv_scores = cross_val_score(svm_clf, X_pca, y, cv=cv, scoring='accuracy', n_jobs=-1)
print("SVM CV Accuracy: {:.2f} (+/- {:.2f})".format(svm_cv_scores.mean(), svm_cv_scores.std() * 2))
y_cv_pred = cross_val_predict(svm_clf, X_pca, y, cv=cv, n_jobs=-1)
print("SVM CV Classification Report:\n", classification_report(y, y_cv_pred, digits=4))
cm = confusion_matrix(y, y_cv_pred, normalize='true')
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y)).plot(cmap='Blues', values_format=".2f")
plt.title("SVM Normalized Confusion Matrix (5-Fold CV)")
plt.show()

# --- Random Forest ---
rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)
rf_cv_scores = cross_val_score(rf_clf, X_pca, y, cv=cv, scoring='accuracy', n_jobs=-1)
print("RF CV Accuracy: {:.2f} (+/- {:.2f})".format(rf_cv_scores.mean(), rf_cv_scores.std() * 2))
y_cv_pred = cross_val_predict(rf_clf, X_pca, y, cv=cv, n_jobs=-1)
print("RF CV Classification Report:\n", classification_report(y, y_cv_pred, digits=4))
cm = confusion_matrix(y, y_cv_pred, normalize='true')
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y)).plot(cmap='Blues', values_format=".2f")
plt.title("Random Forest Normalized Confusion Matrix (5-Fold CV)")
plt.show()
